> **License & Attribution**  
> This notebook is a derivative of [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)  
> by Sebastian Raschka, licensed under the Apache License 2.0.  
> Modifications made by [enhennnn].  
>  
> SPDX-License-Identifier: Apache-2.0  
> For the full license text, see the [LICENSE](LICENSE) file in this repository

## 目录

1. 简单的self-attention
2. 带可训练参数的self-attention
3. 自注意力模块
4. 因果注意力 causal attention
5. 因果注意力模块
6. 多头注意力

&nbsp;
## 1. 简单的self-attention

In [277]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- 只对第二个token计算`注意力输出/上下文向量`

In [278]:
# 第二个token作为查询Query
query = inputs[1]  

# 建立一个空的张量来存储attention分数
attn_scores_2 = torch.empty(inputs.shape[0])

# 第二个token与每个输入的相似性度量计算attention分数
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) 
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [279]:
# 加权式归一化
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum() 
print("tmp Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())


# 自定义softmax归一化
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Naive Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())


# 使用torch的softmax函数进行归一化
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Torch Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

tmp Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)
Naive Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
Torch Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [280]:
# 建立一个零向量存放attention输出/上下文向量
context_vec_2 = torch.zeros(query.shape)

for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


- 推广到对所有token计算`注意力输出/上下文向量`

In [281]:
# 建立个空表,存储每个token与其他token的attention分数
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)


# 使用矩阵乘法计算所有token之间的attention分数
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [282]:
# 使用torch的softmax函数进行归一化
attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [283]:
# 计算所有token的attention输出/上下文向量
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


&nbsp;
## 2. 带可训练参数的self-attention

- `Wq、Wk、Wv`就是可训练参数

In [284]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)


d_in = inputs.shape[1]  # 输入维度
d_out = 2  # 输出维度

In [285]:
torch.manual_seed(123)

# 初始化Wq、Wk、Wv
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print("Query weight matrix W_query:\n", W_query)
print("Key weight matrix W_key:\n", W_key)
print("Value weight matrix W_value:\n", W_value)

Query weight matrix W_query:
 Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
Key weight matrix W_key:
 Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
Value weight matrix W_value:
 Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


- 首先，计算第二个token的`上下文向量`
    
    需要第二个token的`query`和其他token的`key`、`value`矩阵

In [286]:
x_2 = inputs[1]  

# 得到第二个token的query
query_2 = x_2 @ W_query
print("query_2:\n", query_2)

# 计算所有token的key、value
keys = inputs @ W_key 
values = inputs @ W_value
print("keys:\n", keys)
print("values:\n", values)

query_2:
 tensor([0.4306, 1.4551])
keys:
 tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
values:
 tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])


In [287]:
# 第二个token的query对第二个token的key查询
attn_score_2_2 = torch.dot(query_2, keys[1])
print(attn_score_2_2)

# 第二个token的query对所有token的key查询，得到第二个token的attention score
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [288]:
d_k = keys.shape[1]

# 注意力分数除以根号d，使其标准化，再归一化，得到attention weights
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

# 得到注意力输出/上下文向量
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.3061, 0.8210])


&nbsp;
## 3. 自注意力模块

 - 最基础的版本

In [289]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()

        # 初始化Wq、Wk、Wv权重矩阵
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))


    def forward(self, x):
        # 得到Q、K、V矩阵
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        
        # 计算attention score
        attn_scores = queries @ keys.T

        # 标准化和归一化，得到attention weight
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        
        # 计算上下文向量
        context_vec = attn_weights @ values
        return context_vec

In [290]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


 - 使用`Pytorch`的`Linear`层实现

In [291]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        # 初始化Wq、Wk、Wv权重矩阵
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # 得到Q、K、V矩阵
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # 计算attention score
        attn_scores = queries @ keys.T

        # 标准化和归一化，得到attention weight
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        
        # 计算上下文向量
        context_vec = attn_weights @ values
        return context_vec

In [292]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


- `SelfAttention_v1` 和 `SelfAttention_v2` 会给出不同的输出，因为它们使用了不同的初始权重矩阵。

&nbsp;
## 4. 因果注意力 causal attention

 - 因果注意力 = 自注意力 + 掩码

In [293]:
# 将attention scores归一化得到attention weights
sa_v2 = SelfAttention_v2(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

print(attn_weights)

tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


In [294]:
# 设置mask下三角矩阵
seq_len = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(seq_len, seq_len))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [295]:
# 得到masked attention weight，但是得到的结果，每一行之和不等于1
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)


- 如果在 softmax 之后进行掩蔽，它会破坏 softmax 所创建的概率分布。
- softmax 确保所有输出值的总和为 1。
- 如果在 softmax 之后进行掩蔽，就需要重新归一化输出，确保其总和为 1，这会使过程更加复杂，并可能带来意想不到的效果。

In [296]:
# 重新进行归一化
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


- 此前，我们是将attention scores归一化得到attention weights，再加上掩码得到masked attention weight，再归一化

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

- 现在，我们是将attention scores加上掩码得到masked attention scores，再进行归一化得到masked attention weight

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

In [297]:
# 创建一个全1的上三角
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
print(mask)

# 在attention scores加掩码，布尔量为正的地方变为负无穷
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [298]:
# masked attention scores归一化
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


- 引入dropout防止过拟合

    未被dropout的值进行放大，弥补数据缺失的影响。

    放大系数为：`1 / (1 - dropout_rate)`

In [299]:
torch.manual_seed(123)

dropout = torch.nn.Dropout(0.1) 
attn_weights_drop = dropout(attn_weights)
print(attn_weights_drop)

tensor([[1.1111, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4889, 0.6222, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3144, 0.3978, 0.3989, 0.0000, 0.0000, 0.0000],
        [0.2515, 0.2866, 0.2870, 0.2860, 0.0000, 0.0000],
        [0.2114, 0.2248, 0.0000, 0.2219, 0.2278, 0.0000],
        [0.1564, 0.1905, 0.1909, 0.1908, 0.1954, 0.1871]],
       grad_fn=<MulBackward0>)


&nbsp;
## 5. 因果注意力模块

- 在前面自注意力模块的基础上，加上掩码和dropout

In [300]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, seq_len, dropout, qkv_bias=False):
        super().__init__()

        # 初始化Wq、Wk、Wv权重矩阵
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register_buffer跟随模型在不同设备上,并且不进行更新
        self.register_buffer('mask', torch.triu(torch.ones(seq_len, seq_len), diagonal=1))

    def forward(self, x):
        # 得到seq_len，不写在__init__里是因为seq_len在每个batch中可能不同，避免初始化模型时就限制
        _, seq_len, _ = x.shape 

        # 得到Q、K、V矩阵
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # 计算attention scores
        attn_scores = queries @ keys.transpose(1,2)

        # 计算masked attention scores
        attn_scores.masked_fill_(self.mask.bool()[:seq_len, :seq_len], -torch.inf)
        
        # 标准化和归一化，得到attention weight
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        
        # 对attention weights进行dropout
        attn_weights = self.dropout(attn_weights)

        # 计算上下文向量
        context_vec = attn_weights @ values
        return context_vec

In [301]:
torch.manual_seed(123)

# batch的形状是(batch_size, seq_len, hidden_dim)
batch = torch.stack((inputs, inputs), dim=0)
seq_len = batch.shape[1]

ca = CausalAttention(d_in, d_out, seq_len, 0.1)
context_vecs = ca(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5021,  0.2462],
         [-0.6527,  0.0064],
         [-0.7000, -0.0702],
         [-0.6305, -0.0936],
         [-0.6140, -0.1090],
         [-0.5888, -0.1201]],

        [[ 0.0000,  0.0000],
         [-0.6527,  0.0064],
         [-0.4296,  0.0040],
         [-0.6305, -0.0936],
         [-0.6140, -0.1090],
         [-0.5888, -0.1201]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


&nbsp;
## 6. 多头注意力

- 重复调用单头注意力

In [302]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, seq_len, dropout, num_heads, qkv_bias=False):
        super().__init__() 

        # 调用因果注意力模块实现，将同一份数据做num_heads遍，串行执行
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, seq_len, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [303]:
torch.manual_seed(123)

seq_len = batch.shape[1]
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, seq_len, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

- 并行计算

    Pytorch的底层源码中，`“连续”`有严格的数学定义：`“逻辑上相邻的元素，在物理地址上也必须相邻”`

    `view()`不改变数据在内存中的存放顺序，只改变切法。只能按物理结构顺序读，不允许跳着读。

    `transpose()`不改变数据在内存中的存放数据，但是改变读取步长，导致逻辑上相邻的元素在物理地址上不再相邻。因此PyTorch的`.is_contiguous()`返回`False`（即呈现非连续状态）。

    `contiguous()`在内存中重新开辟新空间，拷贝数据，使得数据在物理和逻辑都相邻。

    `reshape()`先判断.is_contiguous，如果连续，执行view()，如果不连续，执行contiguous().view()。

In [304]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, seq_len, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # 确保d_out可以被num_heads整除
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 输出前再经过一层线性层
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask",torch.triu(torch.ones(seq_len, seq_len),diagonal=1))

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # 计算Q、K、V整个矩阵
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 将Q、K、V拆分成head*head_dim
        # view()不改变数据在内存中的存放顺序，只改变切法
        # view()只按物理结构顺序读，不允许跳着读
        # (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, num_heads, head_dim)
        keys = keys.view(batch_size, seq_len, self.num_heads, self.head_dim) 
        values = values.view(batch_size, seq_len, self.num_heads, self.head_dim)
        queries = queries.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # 维度转换，方便后续运算
        # transpose()不改变数据在内存中的存放数据，但是改变读取步长，导致逻辑上相邻的元素在物理地址上不再相邻
        # 因此PyTorch的.is_contiguous()返回False（即呈现非连续状态）

        # (batch_size, seq_len, num_heads, head_dim) -> (batch_size, num_heads, seq_len, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 计算attention scores
        # queries的shape：(batch_size, num_heads, seq_len, head_dim)
        # keys的shape：(batch_size, num_heads, seq_len, head_dim)
        # attn_scores的shape：(batch_size, num_heads, seq_len, seq_len)
        attn_scores = queries @ keys.transpose(2, 3) 

        # 设置掩码
        mask_bool = self.mask.bool()[:seq_len, :seq_len]
        # 得到masked attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 将masked attention scores标准化再归一化，得到masked attention weights
        # softmax不会改变张量的形状，此时的attn_weights的shape：(batch_size, num_heads, seq_len, seq_len)，与attn_scores一致
        # keys.shape[-1]就是head_dim
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # dropout防止过拟合
        attn_weights = self.dropout(attn_weights)

        # 将维度转换回去
        # (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 

        # 因为前面使用了transpose，使得数据在内存中逻辑不相邻
        # 而view()只按物理结构顺序读，不允许跳着读
        # contiguous()在内存中重新开辟新空间，拷贝数据，使得数据在物理和逻辑都相邻
        context_vec = context_vec.contiguous().view(batch_size, seq_len, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [ ]:
torch.manual_seed(123)

batch_size, seq_len, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(d_in, d_out, seq_len, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
